***Set Up***

The code below will display the inputs and outputs which will be later needed to run the main codes. 
The input device is where the guitar will be connected to the PC and the output device is where the sound will come out. 

-Miguel-


In [ ]:
import sounddevice as sd; print(sd.query_devices())

   0 Microsoft Sound Mapper - Input, MME (2 in, 0 out)
>  1 Microphone (Realtek(R) Audio), MME (4 in, 0 out)
   2 Microsoft Sound Mapper - Output, MME (0 in, 2 out)
<  3 Speakers (Realtek(R) Audio), MME (0 in, 2 out)
   4 Primary Sound Capture Driver, Windows DirectSound (2 in, 0 out)
   5 Microphone (Realtek(R) Audio), Windows DirectSound (4 in, 0 out)
   6 Primary Sound Driver, Windows DirectSound (0 in, 2 out)
   7 Speakers (Realtek(R) Audio), Windows DirectSound (0 in, 2 out)
   8 Speakers (Realtek(R) Audio), Windows WASAPI (0 in, 2 out)
   9 Microphone (Realtek(R) Audio), Windows WASAPI (2 in, 0 out)
  10 Headphones 1 (Realtek HD Audio 2nd output with SST), Windows WDM-KS (0 in, 2 out)
  11 Headphones 2 (Realtek HD Audio 2nd output with SST), Windows WDM-KS (0 in, 2 out)
  12 PC Speaker (Realtek HD Audio 2nd output with SST), Windows WDM-KS (2 in, 0 out)
  13 Microphone (Realtek HD Audio Mic input), Windows WDM-KS (2 in, 0 out)
  14 Microphone 1 (Realtek HD Audio Mic input with SS

In [ ]:
import sounddevice as sd      # library to access microphone + speakers in real time
import numpy as np           # numerical operations on arrays (signals)
import time                  # used to keep program running

# -----------------------------
# SYSTEM PARAMETERS
# -----------------------------

fs = 48000        # sampling rate (samples per second)
blocksize = 128   # number of samples processed per block (~2.67 ms of audio)

# Choose which devices to use (from sd.query_devices())
input_device = 15     # microphone
output_device = 12    # headphones

# -----------------------------
# HPF PARAMETERS + STATE
# -----------------------------

a = 0.6       # filter coefficient
# smaller 'a' → more aggressive high-pass (removes more low-frequency content)

prev_x = 0.0    # stores x[n-1] (previous input sample)
prev_y = 0.0    # stores y[n-1] (previous output sample)

# These are REQUIRED because this is a recursive system (has memory)

PortAudioError: Error opening Stream: Invalid number of channels [PaErrorCode -9998]

In [ ]:


# -----------------------------
# HIGH-PASS FILTER FUNCTION
# -----------------------------

def hpf(x):
    """
    Applies a first-order high-pass filter to a block of samples.

    Input:
        x → array of samples (current block)

    Output:
        y → filtered array
    """
    global prev_x, prev_y, a   # we need access to previous samples

    y = np.zeros_like(x)       # allocate output array (same size as input)

    # Loop through each sample in the block
    for n in range(len(x)):

        # Difference equation:
        # y[n] = a ( y[n-1] + x[n] - x[n-1] )
        #
        # Interpretation:
        # - x[n] - x[n-1] → removes slow changes (low frequencies)
        # - + y[n-1]      → keeps system memory
        # - multiplied by a → controls strength of filtering

        y[n] = a * (prev_y + x[n] - prev_x)

        # Update memory for next sample
        prev_x = x[n]
        prev_y = y[n]

    return y

# -----------------------------
# REAL-TIME CALLBACK FUNCTION
# -----------------------------

def callback(indata, outdata, frames, time_info, status):
    """
    This function runs automatically every time a block arrives.

    indata  → input samples from microphone
    outdata → buffer where we must write output samples
    """

    if status:
        print(status)   # print any audio warnings/errors

    # Extract first input channel (mono signal)
    x = indata[:, 0]

    # Apply high-pass filter
    y = hpf(x)

    # Send processed signal to both left and right headphone channels
    outdata[:, 0] = y
    outdata[:, 1] = y

# -----------------------------
# STREAM SETUP
# -----------------------------

stream = sd.Stream(
    device=(input_device, output_device),  # (input, output)
    samplerate=fs,                         # sampling rate
    blocksize=blocksize,                   # block size
    channels=(1, 2),                       # 1 input channel, 2 output channels
    dtype='float32',                       # audio sample format
    callback=callback                      # function to run every block
)

# Start audio processing
stream.start()

print("Running HPF... stop with interrupt button")

# -----------------------------
# KEEP PROGRAM ALIVE
# -----------------------------

# Without this loop, the program would exit immediatel
while True:
    time.sleep(0.1)

Running HPF... stop with interrupt button


KeyboardInterrupt: 